In [ ]:
# CELL 1: MOUNT DRIVE
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

BASE = "/content/drive/MyDrive/fraud_detection"

train = pd.read_csv(f"{BASE}/fraudTrain.csv/fraudTrain.csv", index_col=0)
test = pd.read_csv(f"{BASE}/fraudTest.csv/fraudTest.csv", index_col=0)

print(train.shape, test.shape)
train.head(3)

In [ ]:
# fraud class imbalance  check : This cell is checking how much fraud exists compared to normal transactions.
# 0 = Normal 1 = fraud separately for train & test from 1,000,000 999,000 are normal -> 1,000 can be fraud (0.1%) this is class imbalance as no of 0's >> 1's
# model becomes biased towards majority of the class  


def fraud_rate(df: pd.DataFrame, name: str) -> None:    # fucn accepting dataset & doesnt return anything 
    counts = df["is_fraud"].value_counts()              # "Count how many times each value appears."
    rate = counts.get(1, 0) / len(df) * 100
    print(f"{name}: {len(df):,} rows | fraud={counts.get(1,0):,} ({rate:.3f}%)")

fraud_rate(train, "train")
fraud_rate(test, "test")

# fraud is very small So your model is going to see far more normal transactions than fraudulent ones 
# If almost everything is 0, the model can become very good at predicting 0 and still be bad at finding 1. 

In [ ]:
# parse datetimes
# DATA PREPROCESSING -> Taking raw data and putting it into a form that the computer can actually work with.

for df in (train, test):
    df["trans_date_trans_time"] = pd.to_datetime(df["trans_date_trans_time"])
    df["dob"] = pd.to_datetime(df["dob"])

'''
2019-01-01 00:00:18 → datetime
1988-03-09          → datetime 
converting the str to datetime obj's
Transaction suddenly happens at 3 AM instead 2pm suppicious 
'''

In [ ]:
# FEAT ENGG -> Take raw transaction information and create new columns that may help the model recognize fraud. 

import numpy as np

def haversine_km(lat1, lon1, lat2, lon2) -> np.ndarray:
    # * distance between cardholder home and merchant, in km
    # * Stripe's blog: signals like "is this IP/location typical for this card" are
    # * some of the highest-value fraud features — this is our version of that
    r = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return r * 2 * np.arcsin(np.sqrt(a)) # This returns the calculated distance in kilometers.


def add_distance_feature(df: pd.DataFrame) -> pd.DataFrame:
    df["distance_km"] = haversine_km(df["lat"], df["long"], df["merch_lat"], df["merch_long"])
    return df

# This creates useful information from the transaction timestamp. 
def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    df["hour"] = df["trans_date_trans_time"].dt.hour
    df["day_of_week"] = df["trans_date_trans_time"].dt.dayofweek
    df["age"] = (df["trans_date_trans_time"] - df["dob"]).dt.days // 365
    return df


def add_velocity_features(df: pd.DataFrame) -> pd.DataFrame:
    # * time since this card's previous transaction, and a rolling per-card amount
    # * z-score — proxies for Radar's "previous encounters with a card" signal
    df = df.sort_values(["cc_num", "trans_date_trans_time"]).copy()     # treat each card separately 
    df["prev_trans_time"] = df.groupby("cc_num")["trans_date_trans_time"].shift(1)

# Calculate time since previous transaction
    df["seconds_since_last_trans"] = (
        (df["trans_date_trans_time"] - df["prev_trans_time"]).dt.total_seconds()
    )

# You're replacing them with the median of the column. 
    df["seconds_since_last_trans"] = df["seconds_since_last_trans"].fillna(
        df["seconds_since_last_trans"].median()
    )

# Amount behavior per card 
    card_amt_mean = df.groupby("cc_num")["amt"].transform("mean")

# Standard deviation tells you how spread out the card's transaction amounts usually are. 
# 105 rs isnt suprising but 10,000 is very unsual 
    card_amt_std = df.groupby("cc_num")["amt"].transform("std").replace(0, 1)

# How unusual is this transaction amount compared with this particular card's normal spending? 
# normally person spends 100rs immediatlyspend 50000
# * How unusual is this value compared with the normal beehaviour 
    df["amt_zscore_per_card"] = (df["amt"] - card_amt_mean) / card_amt_std

    return df.drop(columns=["prev_trans_time"])

'''
Transactions for each card are placed in each chronological order
Card A
10:00 → ₹100
10:30 → ₹200
11:15 → ₹150

Card B
09:00 → ₹500
12:00 → ₹700
this is necesssar to know What happened immediately before this transaction?
'''


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = add_distance_feature(df)
    df = add_time_features(df)
    df = add_velocity_features(df)
    return df


'''
how far the mercan from customer location
Haversine = calculate geographic distance between two latitude/longitude points.
Lat & Long(degree -> radians) are co-ordinates of Earth 

lat       = Pune latitude
long      = Pune longitude

merch_lat = Mumbai latitude
merch_long= Mumbai longitude
             ↓
distance_km = ~120 km

we started with 
    Raw transaction
    │
    ├── lat / long
    ├── merch_lat / merch_long
    ├── transaction time
    ├── DOB
    ├── card number
    └── transaction amount
Then u create 
    distance_km
    hour
    day_of_week
    age
    seconds_since_last_trans
    amt_zscore_per_card
Feautures : These are called feautures A piece of information given to the ML model to help it make a prediction.
'''

In [ ]:
# applying to both sets

train_fe = engineer_features(train)
test_fe = engineer_features(test)

train_fe[["distance_km", "hour", "day_of_week", "age",
          "seconds_since_last_trans", "amt_zscore_per_card", "is_fraud"]].describe()

'''
    describe gives u statistical info 
    Taking raw information and creating better inputs for an ML model.
    mean   ≈ 76.11 km
    min    ≈ 0.02 km
    median ≈ 78.23 km
    max    ≈ 152.12 km 
'''

₹500
₹700
₹400
₹600

Then suddenly:

₹50,000

Your feature engineering could produce something like:

amt_zscore_per_card = very high

At the same time:

seconds_since_last_trans = 30
distance_km = 1,200
hour = 3

Now the ML model receives several signals saying:

"This transaction looks different from this customer's normal behavior."

The model can combine these signals to estimate:

P(fraud) = high  

CELL 1
Load raw train/test
       ↓
CELL 2
Check fraud imbalance
       ↓
CELL 3
Convert dates into datetime
       ↓
CELL 4
Define feature-engineering functions
       ↓
CELL 5 ← YOU ARE HERE
Actually apply those functions
       ↓
Check the resulting features
       ↓
NEXT
Prepare features + target for ML

In [ ]:
# 🧩 Cell 6 — Compare feature behavior for fraud vs normal 

train_fe.groupby("is_fraud")[["distance_km", "amt_zscore_per_card", "seconds_since_last_trans"]].mean()

'''
Do my engineered features actually behave differently for fraud and normal transaction
Normal → 32,538 sec
Fraud  → 21,276 sec

Fraud transactions happen, on average, closer together in time than normal transactions.

That could be useful to the model.
'''

In [ ]:
# Encode categoricals + build feature matrix

FEATURES_NUM = ["amt", "hour", "day_of_week", "age",
                "seconds_since_last_trans", "amt_zscore_per_card", "city_pop"]
FEATURES_CAT = ["category", "gender"]

def build_matrix(df: pd.DataFrame, cat_maps: dict | None = None):
    # * one-hot the low-cardinality categoricals; if cat_maps is given (from train),
    # * reindex test to the same columns so train/test have identical feature shape
    X_cat = pd.get_dummies(df[FEATURES_CAT], drop_first=True) # performs hot encoding - a method that changes text or category names into numbers of 0s and 1s so computers and Machine Learning models can understand them.
    X = pd.concat([df[FEATURES_NUM], X_cat], axis=1)
    if cat_maps is not None:
        X = X.reindex(columns=cat_maps, fill_value=0) # Make the test dataset use exactly the same columns as training. If a column is missing, fill it with 0."
    return X

X_train = build_matrix(train_fe) # which contains the features used to train the model
X_test = build_matrix(test_fe, cat_maps=X_train.columns)  # Make the test dataset use exactly the same columns as training. If a column is missing, fill it with 0."
# **** with exactly the same feature columns as X_train. 

y_train = train_fe["is_fraud"]
y_test = test_fe["is_fraud"]

X_train.shape, X_test.shape

'''
Decides which features the model should use.
Converts categorical text like "male" or "grocery_pos" into numbers using one-hot encoding.
Separates:
X → information the model uses to make predictions
y → the actual answer: fraud or not fraud

"These numerical features are the ones I want to give to the model."
Notice that you aren't giving the model every column from your original dataset.
You're selecting the features you decided are useful.

7. Simple fraud example

Suppose one transaction originally looks like:

amt = ₹5,000
hour = 3
age = 25
category = grocery
gender = F

After encoding, the model might receive something conceptually like:

amt     hour   age   category_grocery   category_shopping   gender_M
5000     3     25          1                  0                 0

'''

In [ ]:
# Baseline: Logistic Regression

from sklearn.linear_model import LogisticRegression     # ur ML model 
from sklearn.preprocessing import StandardScaler        # scale numerical feautures 
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test) # X_test_scaled

# * class_weight="balanced" — never resample the test set, only reweight training loss
logreg = LogisticRegression(class_weight="balanced", max_iter=1000)
logreg.fit(X_train_scaled, y_train) # Your model is learning from:

probs_lr = logreg.predict_proba(X_test_scaled)[:, 1]
preds_lr = (probs_lr >= 0.5).astype(int)

print(classification_report(y_test, preds_lr, digits=3))
print("ROC-AUC:", roc_auc_score(y_test, probs_lr))
print("PR-AUC (average precision):", average_precision_score(y_test, probs_lr))



'''
Baseline = A first model that gives you a reference point.
fit
The scaler learns the mean and standard deviation from the training data.
transform
It uses those learned values to scale the training data.

X_train
  ↓
learn scaling parameters
  ↓
scale X_train
  ↓
X_train_scaled

'''

1. ROC-AUC DefinitionROC-AUC stands for Receiver Operating Characteristic – Area Under the Curve.It measures a model's ability to distinguish between two classes (positive and negative) across all possible probability thresholds.The Curve (ROC): Plots the True Positive Rate (Sensitivity) against the False Positive Rate (1 - Specificity) as you change the prediction threshold from 0 to 1.The Area (AUC): A single score between 0 and 1. A score of 0.5 means random guessing. A score of 1.0 means perfect separation.


2. PR-AUC DefinitionPR-AUC stands for Precision-Recall – Area Under the Curve (also known as Average Precision).It measures the relationship between a model's predictive accuracy for the positive class and its ability to find all positive instances.The Curve (PR): Plots Precision on the y-axis and Recall on the x-axis across all possible probability thresholds.The Area (AUC): A single score between 0 and 1. Unlike ROC, the baseline for a random model is not 0.5; it is equal to the percentage of positive cases in your dataset.

In [ ]:
# Baseline: Logistic Regression

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# * class_weight="balanced" — never resample the test set, only reweight training loss
logreg = LogisticRegression(class_weight="balanced", max_iter=1000)
logreg.fit(X_train_scaled, y_train)

probs_lr = logreg.predict_proba(X_test_scaled)[:, 1]
preds_lr = (probs_lr >= 0.5).astype(int)

print(classification_report(y_test, preds_lr, digits=3))
print("ROC-AUC:", roc_auc_score(y_test, probs_lr))
print("PR-AUC (average precision):", average_precision_score(y_test, probs_lr))

Notice something important:

You use only transform, not fit_transform.

Why?

Because the test set is supposed to represent unseen data.

You don't want the scaler learning anything from the test set.

So:

TRAIN
fit + transform
     ↓
TEST
transform only

This prevents data leakage.

Data leakage means accidentally allowing information from the test data to influence the training process.

Recall = 0.743

This is good to understand.

There were:

2,145 actual fraud transactions

The model caught about:

74.3%

of them.

Approximately:

2,145 × 0.743 ≈ 1,594

So the model detected roughly 1,594 fraud transactions.

It missed roughly:

2,145 - 1,594 ≈ 551

fraud transactions.

Those missed frauds are false negatives

Precision = 0.024

This is the painful part.

Precision is:

"When my model says FRAUD, how often is it actually fraud?"

Your precision is:

2.4%

That's very low.

Imagine the model flags 1,000 transactions as fraud.

Only around:

24

might actually be fraud.

The other approximately:

976

would be legitimate transactions incorrectly flagged.

Those are false positives.

This is a huge issue for a real payment system because you could annoy/block many legitimate customers.

***** f1 score is 4.7% "I'll catch a lot of fraud, but I'll also flag a huge number of legitimate transactions.That's why simply saying:
"My recall is 74%, therefore my model is good"
would be misleading."

88.5% accuracy not soo cool as fraud is rare

11. ROC-AUC = 0.911

You got:

ROC-AUC: 0.9108

ROC-AUC measures how well the model can rank fraud above normal transactions across different thresholds. suggest basically the model has good overall ability to distinguish classes 

***** 1. ROC-AUC = 0.911 (Excellent Overall Sorting)What it is: The Area Under the Receiver Operating Characteristic curve.What 0.911 means: If you pick one random positive case and one random negative case, your model will correctly give the positive case a higher probability score 91.1% of the time.The Catch: It looks amazing because it is very good at identifying the majority class (the negative cases).2. PR-AUC = 0.139 (Poor Target Detection)What it is: The Area Under the Precision-Recall curve.What 0.139 means: When your model predicts that an event is positive, it is wrong most of the time (Low Precision). It is also missing a lot of the actual positive cases (Low Recall).The Reality: In an imbalanced dataset, PR-AUC is the true test of performance. A score of 0.139 means the model is struggling to find the "needle in the haystack."A Real-World Analogy: Email Spam FilterImagine a dataset of 1,000 emails where only 10 are actually spam (positive class) and 990 are safe (negative class).ROC-AUC (0.911): The model is great at knowing that safe emails are safe. If it just labels everything as safe, it is already 99% accurate. ROC-AUC gets inflated by this massive success on the safe emails.PR-AUC (0.139): Out of the 10 real spam emails, the model might only find 2 (Low Recall). Meanwhile, it accidentally flags 15 safe emails as spam (Low Precision). The PR-AUC exposes this failure.



# xgboost

!pip install -q xgboost

from xgboost import XGBClassifier

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    eval_metric="aucpr",
    random_state=42,
)
xgb.fit(X_train, y_train)

probs_xgb = xgb.predict_proba(X_test)[:, 1]
preds_xgb = (probs_xgb >= 0.5).astype(int)

print(classification_report(y_test, preds_xgb, digits=3))
print("ROC-AUC:", roc_auc_score(y_test, probs_xgb))
print("PR-AUC (average precision):", average_precision_score(y_test, probs_xgb))

********** A fraudulent transaction might look suspicious because of a combination:

unusual amount + unusual time + very short gap from previous transaction + certain category + particular customer behavior

XGBoost is much better at learning these non-linear relationships and combinations between features.

In [ ]:
# xgboost

# !pip install -q xgboost

from xgboost import XGBClassifier

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6, # Controls how deep each decision tree can become.
    learning_rate=0.1, # controls how strongly each tree contributes 0.1 means each tree makes a relatively controlled contribution rather than completely changing the prediction.
    scale_pos_weight=scale_pos_weight, # handles class imbalnce gives importance the rare positive class:
    eval_metric="aucpr", # This tells XGBoost to use PR-AUC as its evaluation metric. as Because fraud detection has a highly imbalanced dataset.
    random_state=42, # Makes the model's randomness reproducible. for consistent results.
)
xgb.fit(X_train, y_train) # ** training step where he actually learns "What combinations of transaction characteristics tend to correspond to fraud?"

probs_xgb = xgb.predict_proba(X_test)[:, 1]
preds_xgb = (probs_xgb >= 0.5).astype(int)

print(classification_report(y_test, preds_xgb, digits=3))
print("ROC-AUC:", roc_auc_score(y_test, probs_xgb))
print("PR-AUC (average precision):", average_precision_score(y_test, probs_xgb))

pecion =38%
When XGBoost says "this transaction is fraud", about 38% of those flagged transactions are actually fraud.

The other ~62% are legitimate transactions that got flagged.

Those are false positives.



**** 12. PR-AUC = 0.909
PR-AUC = 0.9086

This is probably the most exciting result in this cell for your fraud project.

Your Logistic Regression had:

PR-AUC = 0.1386

XGBoost:

PR-AUC = 0.9086

That's a gigantic jump.

It means XGBoost is doing far better at ranking fraud while maintaining useful precision/recall behavior across thresholds.

*********14. Real-world fraud example

Imagine Razorpay processes:

100,000 transactions.

Suppose:

1,000 = fraud
99,000 = legitimate

Your model might catch around:

953 fraud

because recall ≈ 95.3%.

But because precision is 38%, the model also flags legitimate transactions.

So the real business question becomes:

"Can we change the threshold so that we catch enough fraud while reducing legitimate customers being blocked?

In [ ]:
# prescision recall curve & threshold selection - "Where should I set the threshold for deciding that a transaction is fraud?"
# ** So threshold selection is a business decision informed by the model, not just a mathematical choice.
from sklearn.metrics import precision_recall_curve # helps to calculates precision and recall at many different classification thresholds.
import matplotlib.pyplot as plt

precisions, recalls, thresholds = precision_recall_curve(y_test, probs_xgb)

plt.figure(figsize=(6,5))
plt.plot(recalls, precisions)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("XGBoost — Precision-Recall Curve")
plt.grid(True)
plt.show()

'''Beginner intuition

Imagine your model flags 100 transactions.

If precision is 11.25%, approximately:

11.25 → actually fraud
88.75 → actually legitimate

The fraud you successfully stop has enough economic value to roughly compensate for the profit lost from the legitimate transactions you wrongly flag.

So 11.25% is your simplified break-even point.

7. Why ₹1,775 vs ₹225 matters

This is the core business insight.

Your simplified assumptions say:

False positive

You incorrectly flag a legitimate customer:

Cost ≈ ₹225
False negative

You allow fraud through:

Cost ≈ ₹1,775

So:

Missed fraud is ~7.9× more expensive
than blocking a legitimate sale.'''



6. Why does precision fall near recall = 1?

This is the really important intuition.

Suppose there are 2,145 fraud transactions.

To get extremely close to:

Recall = 100%

you basically have to become very aggressive about flagging transactions.

Eventually, the model starts saying:

"I'm going to flag transactions that aren't as suspicious either, because I don't want to miss any fraud."

That catches more fraud.

But it also catches more legitimate transactions.

Therefore:

Recall ↑
Precision ↓

This is the fundamental precision-recall tradeoff

In [ ]:

# ****** 0. Connecting this to your XGBoost result 🔥

# This is where your previous cells become interesting.

# Your XGBoost at threshold 0.5 gave:

# Precision = 38.0%
# Recall = 95.3%

# Your calculated break-even precision is:

# 11.25%

# So:

# 38.0% > 11.25%

# ***** Given these specific assumptions about transaction value, margin, and fraud cost, 11.25% is the simplified precision at which the economics break even.


In [ ]:
# Finding exact threshold - It's finding the closest available precision value in the curve.

import numpy as np

target_precisions = [0.90, 0.70, 0.50, 0.30, breakeven_precision]

for tp in target_precisions:
    idx = np.argmin(np.abs(precisions[:-1] - tp)) # **The [:-1] keeps the precision array aligned with the available thresholds.
    print(f"precision≈{precisions[idx]:.3f}  recall={recalls[idx]:.3f}  threshold={thresholds[idx]:.4f}")

# "I evaluated the precision-recall tradeoff and selected operating thresholds based on business costs. Under my assumed economics, the break-even precision was 11.25%, while different thresholds provide different precision/recall operating points."

In [ ]:
# RADAR-style aggregate spike detector
# Aggregate fraud rate by merchant/category over time

THRESHOLD = 0.35

test_fe = test_fe.copy()
test_fe["fraud_pred"] = (probs_xgb >= THRESHOLD).astype(int)
test_fe["fraud_score"] = probs_xgb

# * bucket into daily windows per category (merchant-level would be sparser;
# * category is a cleaner segment to demo spike detection on)
test_fe["date"] = test_fe["trans_date_trans_time"].dt.date

# ** Group transactions by category and date, then count transactions and fraud.
daily_by_category = (
    test_fe.groupby(["category", "date"])
    .agg(n_trans=("is_fraud", "size"), n_fraud=("is_fraud", "sum"))
    .reset_index()
)
daily_by_category["fraud_rate"] = daily_by_category["n_fraud"] / daily_by_category["n_trans"]

daily_by_category.head()

In [ ]:
# Rolling baseline + z-score spike flag

def flag_spikes(group: pd.DataFrame, window: int = 7, k: float = 2.5) -> pd.DataFrame:
    # * rolling mean/std of fraud_rate as the "expected" baseline for this category
    # * (our lightweight stand-in for Uber Orbit's time-series decomposition)
    group = group.sort_values("date").copy()
    group["baseline_mean"] = group["fraud_rate"].rolling(window, min_periods=3).mean()
    group["baseline_std"] = group["fraud_rate"].rolling(window, min_periods=3).std().replace(0, np.nan)
    group["z_score"] = (group["fraud_rate"] - group["baseline_mean"]) / group["baseline_std"]
    group["is_spike"] = group["z_score"] > k
    return group

spikes_df = (
    daily_by_category.groupby("category", group_keys=False)
    .apply(flag_spikes)
)

spikes_df[spikes_df["is_spike"] == True].sort_values("z_score", ascending=False).head(15)

In [ ]:
# Inspect the z-score distribution

print(spikes_df["z_score"].describe())
print("\nRows with baseline computed:", spikes_df["baseline_mean"].notna().sum(), "/", len(spikes_df))
spikes_df.sort_values("z_score", ascending=False).head(10)[
    ["category", "date", "n_trans", "n_fraud", "fraud_rate", "baseline_mean", "baseline_std", "z_score"]
]

In [ ]:
test_fe["week"] = test_fe["trans_date_trans_time"].dt.to_period("W").apply(lambda p: p.start_time.date())

weekly_by_category = (
    test_fe.groupby(["category", "week"])
    .agg(n_trans=("is_fraud", "size"), n_fraud=("is_fraud", "sum"))
    .reset_index()
)
weekly_by_category["fraud_rate"] = weekly_by_category["n_fraud"] / weekly_by_category["n_trans"]

def flag_spikes_weekly(group: pd.DataFrame, window: int = 4, k: float = 2.0) -> pd.DataFrame:
    group = group.sort_values("week").copy()
    group["baseline_mean"] = group["fraud_rate"].rolling(window, min_periods=3).mean()
    group["baseline_std"] = group["fraud_rate"].rolling(window, min_periods=3).std().replace(0, np.nan)
    group["z_score"] = (group["fraud_rate"] - group["baseline_mean"]) / group["baseline_std"]
    return group

weekly_spikes = (
    weekly_by_category.groupby("category", group_keys=False)
    .apply(flag_spikes_weekly)
)

print(weekly_spikes["z_score"].describe())
weekly_spikes.sort_values("z_score", ascending=False).head(10)[
    ["category", "week", "n_trans", "n_fraud", "fraud_rate", "baseline_mean", "baseline_std", "z_score"]
]

In [ ]:
# * since z-scores don't cleanly separate real bursts (max z=1.5 — no coordinated
# * attack waves in this simulated dataset), we pivot from binary spike detection
# * to a relative risk ranking: surface the top-N category-weeks by z-score as
# * "elevated risk" for analyst attention, rather than requiring a hard statistical cutoff

weekly_spikes["risk_percentile"] = weekly_spikes["z_score"].rank(pct=True)

TOP_N_PCT = 0.95  # top 5% of category-weeks by z-score, surfaced for review

elevated_risk = (
    weekly_spikes[weekly_spikes["risk_percentile"] >= TOP_N_PCT]
    .sort_values("z_score", ascending=False)
    .reset_index(drop=True)
)

print(f"Elevated-risk segments flagged: {len(elevated_risk)} / {len(weekly_spikes)}")
elevated_risk[["category", "week", "n_trans", "n_fraud", "fraud_rate", "baseline_mean", "z_score"]]

In [ ]:
flagged = test_fe[test_fe["fraud_pred"] == 1].copy()

def top_patterns(df: pd.DataFrame, cols: list[str], top_n: int = 10) -> pd.DataFrame:
    # * simple crosstab-based pattern surfacing — a lightweight stand-in for
    # * RADAR's FP-Growth pattern mining, appropriate for our scale and timeframe
    combo = df.groupby(cols).size().reset_index(name="count")
    return combo.sort_values("count", ascending=False).head(top_n)

print("Top category + hour patterns among flagged transactions:")
display(top_patterns(flagged, ["category", "hour"]))

print("\nAmount z-score bucket among flagged transactions:")
flagged["amt_z_bucket"] = pd.cut(flagged["amt_zscore_per_card"], bins=[-5, 1, 2, 3, 5, 100])
display(flagged["amt_z_bucket"].value_counts())

In [ ]:
import matplotlib.pyplot as plt

importances = pd.Series(xgb.feature_importances_, index=X_train.columns).sort_values(ascending=False)

print(importances.head(10))

plt.figure(figsize=(8,5))
importances.head(10).plot(kind="barh")
plt.gca().invert_yaxis()
plt.title("XGBoost — Top 10 Feature Importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

In [ ]:
import pickle
import json

# * bundle everything the FastAPI layer needs: model, scaler, feature column order,
# * chosen threshold, and the Layer-2 baseline stats — so the API doesn't need
# * to recompute anything from raw data at request time

artifact = {
    "model": xgb,
    "scaler": scaler,
    "feature_columns": list(X_train.columns),
    "threshold": THRESHOLD,
    "breakeven_precision": breakeven_precision,
}

with open("/content/drive/MyDrive/fraud_detection/fraud_model.pkl", "wb") as f:
    pickle.dump(artifact, f)

# * Layer 2 baseline stats per category, so the API can rank new weekly data
# * against known historical baselines without needing the full training set loaded
category_baselines = (
    weekly_by_category.groupby("category")["fraud_rate"]
    .agg(["mean", "std"])
    .reset_index()
    .rename(columns={"mean": "baseline_mean", "std": "baseline_std"})
)
category_baselines.to_json(
    "/content/drive/MyDrive/fraud_detection/category_baselines.json",
    orient="records"
)

print("Saved fraud_model.pkl and category_baselines.json")
print(category_baselines)